# GazeMedSeg × webcam-gaze — Kvasir-SEG Dice (one-click)

Computes the **webcam-gaze segmentation Dice** for paper §8 by dropping our webcam
fixations into GazeMedSeg's unchanged Kvasir-SEG pipeline.

**Runtime → Change runtime type → GPU** before running. Then *Runtime → Run all*.

You will be asked to upload the 4 batch CSVs you collected
(`kvasir_fixation_webcam_p1of4.csv` … `p4of4.csv`).

> Best-effort automation from the documented repo layout. The two cells tagged
> **VERIFY** may need a path tweak on first run (the repo hardcodes a dataset
> root); everything else is mechanical.


## 0. GPU check


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime type to GPU'


## 1. Clone GazeMedSeg + install deps


In [ ]:
!git clone https://github.com/med-air/GazeMedSeg.git
%cd GazeMedSeg
!ls


In [ ]:
# Colab ships torch/torchvision; add the rest their pipeline needs.
!pip -q install gdown opencv-python-headless scipy scikit-image pandas matplotlib
# dense CRF for the pseudo-mask step:
!pip -q install pydensecrf || pip -q install git+https://github.com/lucasb-eyer/pydensecrf.git

!pip -q install monai  # GazeMedSeg's UNet (BasicUNet) lives in monai

!pip -q install pydicom  # datasets/__init__ imports the NCI loader (needs pydicom) even for kvasir


## 2. Get the preprocessed Kvasir-SEG dataset
GazeMedSeg's preprocessed datasets (correct image/mask layout + train/test split).


In [ ]:
# Preprocessed datasets folder from the GazeMedSeg README.
!gdown --folder 'https://drive.google.com/drive/folders/1XjgQ27R8zT8ymOTXohgl8HXntPEUbIXj' -O ./data
print('--- layout ---')
!find ./data -maxdepth 3 | head -40


**VERIFY:** set `DATA_ROOT` to the Kvasir-SEG dataset root printed above
(the folder that directly contains the `images/` and `masks/` subfolders).


In [ ]:
import os, glob, zipfile
for z in glob.glob('data/**/*.zip', recursive=True):
    try:
        with zipfile.ZipFile(z) as f: f.extractall(os.path.dirname(z)); print('unzipped', z)
    except Exception as e: print('skip', z, e)
# DATA_ROOT = the KVASIR folder (has images/+masks/); prefer 'kvasir' over NCI-ISBI
cands=[os.path.dirname(p) for p in glob.glob('data/**/images', recursive=True) if os.path.isdir(os.path.dirname(p)+'/masks')]
DATA_ROOT=next((c for c in cands if 'kvasir' in c.lower()), cands[0] if cands else './data')
print('DATA_ROOT =', DATA_ROOT)
print('images:', len(glob.glob(os.path.join(DATA_ROOT,'images','*'))),
      '| masks:', len(glob.glob(os.path.join(DATA_ROOT,'masks','*'))),
      '| splits:', glob.glob(os.path.join(DATA_ROOT,'*.txt')))


## 3. Upload + concatenate your 4 webcam CSVs
They are already in GazeMedSeg's fixation schema, so they drop straight in.


In [ ]:
from google.colab import files
import pandas as pd
print('Upload kvasir_fixation_webcam_p1of4.csv … p4of4.csv')
up = files.upload()
parts = sorted(up.keys())
df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df.to_csv('kvasir_fixation.csv', index=False)
print('merged', len(df), 'fixations over', df.IMAGE.nunique(), 'images')
# the gaze-annotation notebook reads ../../GazeMedSeg/kvasir_fixation.csv
!mkdir -p GazeMedSeg && cp kvasir_fixation.csv GazeMedSeg/kvasir_fixation.csv


## 4. (zero-GPU) Full-set go/no-go
Before spending GPU: does the *full* webcam gaze localise the polyp better than
chance? `peak_in_GT` ≫ chance and well above the 23% pilot → proceed.
This mirrors the pilot diagnostic on all collected images.


In [ ]:
import numpy as np, os, glob
from collections import defaultdict
from scipy.ndimage import gaussian_filter
from PIL import Image
fix=defaultdict(list)
for _,r in df.iterrows():
    fix[r['IMAGE']].append((float(r['CURRENT_FIX_X']),float(r['CURRENT_FIX_Y']),float(r['CURRENT_FIX_DURATION'])))
def mask_path(name):
    p=os.path.join(DATA_ROOT,'masks',name)
    return p if os.path.exists(p) else os.path.join(DATA_ROOT,'masks',os.path.splitext(name)[0]+'.jpg')
peak=0; masses=[]; chance=[]; n=0
for name,fs in fix.items():
    mp=mask_path(name)
    if not os.path.exists(mp): continue
    gt=(np.array(Image.open(mp).convert('L'))>127).astype(float); H,W=gt.shape
    if gt.sum()==0: continue
    hm=np.zeros((H,W))
    for x,y,d in fs: hm[min(H-1,max(0,int(y*H))),min(W-1,max(0,int(x*W)))]+=d
    hm=gaussian_filter(hm,70)
    if hm.max()<=0: continue
    py,px=np.unravel_index(hm.argmax(),hm.shape)
    peak+=int(gt[py,px]>0); masses.append(float((hm*gt).sum()/hm.sum())); chance.append(gt.mean()); n+=1
print(f'images scored: {n}')
print(f'peak-in-GT : {100*peak/max(1,n):.0f}%   (pilot was 23%, EyeLink ~80%)')
print(f'mass-in-GT : mean {np.mean(masses):.3f}   chance {np.mean(chance):.3f}')


## 5. Generate pseudo-masks from YOUR gaze
Runs GazeMedSeg's own gaze→heatmap→CRF notebook, now reading your CSV.


In [ ]:
# Point GazeMedSeg's gaze-annotation notebook at our dataset + CSV, then run
# it headless so it writes gaze/ heatmaps + CRF pseudo-masks from OUR webcam gaze.
import os, json, shutil
abs_root = os.path.abspath(DATA_ROOT)
os.makedirs('GazeMedSeg', exist_ok=True)
shutil.copy('kvasir_fixation.csv', 'GazeMedSeg/kvasir_fixation.csv')  # notebook reads ../../GazeMedSeg/kvasir_fixation.csv
nbp = 'notebooks/gaze_annotation/generate_gaze_annotation_kvasir.ipynb'
nb = json.load(open(nbp))
for c in nb['cells']:
    if c['cell_type'] == 'code':
        c['source'] = [s.replace('PUT YOUR PATH OF KVASIR-SEG DATASET HERE', abs_root) for s in c['source']]
json.dump(nb, open(nbp, 'w'))
print('patched dataset root ->', abs_root)
!jupyter nbconvert --to notebook --execute --inplace \
  notebooks/gaze_annotation/generate_gaze_annotation_kvasir.ipynb \
  --ExecutePreprocessor.timeout=1800
!find . -path '*gaze*' -name '*.jpg' | head


**VERIFY:** if the cell above errors on a path, open
`notebooks/gaze_annotation/generate_gaze_annotation_kvasir.ipynb`, set its dataset
root to `DATA_ROOT`, and re-run. It should create `{DATA_ROOT}/gaze/heatmap/` and
`{DATA_ROOT}/gaze/crf_compat*/`.


## 6. Train on the gaze pseudo-masks (GPU)
Their 2-level U-Net; ~15k iters (~1–2 h on a Colab T4).


In [ ]:
# GazeMedSeg's 2-level U-Net on OUR gaze pseudo-masks. pseudo_root auto-resolves
# to {root}/gaze. -bs 2 + expandable_segments fits a free Colab T4 (16GB);
# paper used bs 8 (needs a bigger GPU); bs 2 is a free-tier concession.
# the feature-propagation einsum is the memory bottleneck at 224^2.
import os
ROOT = os.path.abspath(DATA_ROOT)
print('root:', ROOT,
      '| gaze/crf_compat1:', os.path.isdir(ROOT+'/gaze/crf_compat1'),
      '| train.txt:', os.path.exists(ROOT+'/train.txt'),
      '| test.txt:', os.path.exists(ROOT+'/test.txt'))

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python run.py -m gaze_sup --data kvasir --model unet -bs 2 \
  --exp_path ./exp --root "{ROOT}" \
  --spatial_size 224 --in_channels 3 --opt sgd --lr 1e-2 --lr_min 1e-4 \
  --lr_scheduler cos --max_ite 15000 --num_levels 2 --cons_mode prop \
  --cons_weight 3 --data_size_rate 1 --device 0 --seed 0 --num_worker 2


## 7. Read the Dice
Their training logs the test Dice. Grab the final/best value.


In [ ]:
!grep -riE 'dice' --include='*.log' --include='*.txt' . 2>/dev/null | tail -30
# also check any results json / tensorboard dir printed by the script above


## 8. Fill the paper
Put the test Dice into Table `tab:downstream` (the **TODO** cell) and compare to
EyeLink **77.80** and full-mask **82.12**. Tell Claude the number; it writes the
results paragraph (framed as utility, not a ranking).
